# CARISMA - Téléchargement et extraction des données magnétométriques

***

**Tutoriel :** Ce tutoriel explique comment extraire les données magnétométriques CARISMA depuis le portail de données ouvertes.  
**Mission et instrument :** CARISMA (Canadian Array for Realtime Investigations of Magnetic Activity)  
**Objectif scientifique :** Mesurer le champ magnétique terrestre pour étudier les événements de météorologie spatiale tels que les tempêtes géomagnétiques et les sous-tempêtes.  
**Configuration requise :** Accès à Internet.
**Niveau du tutoriel :** Intermédiaire

Les données CARISMA sont disponibles au format CSV et en données brutes sur le portail de données ouvertes de l'ASC. Les données brutes se trouvent [ici](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/carisma/) et les fichiers CSV sont disponibles [ici](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/carisma_csv/). De plus, les données CARISMA sont également hébergées par l'Université de l'Alberta sur [carisma.ca](https://www.carisma.ca/).

# Partie 1 : Téléchargement des données CARISMA

Les données CARISMA/CANOPUS sont hébergées sur le [CSA Open Data Portal](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub). Les fichiers peuvent être téléchargés par scrapage de contenu.

## 1.1 Téléchargement programmatique (scraping HTTP/HTTPS)

### Structure du répertoire

```
/users/OpenData_DonneesOPuvertes/pub/
|
|-- carisma_csv/                                    <-- données magnétométriques CARISMA (csv)
|
|-- CANOPUS_CSV/                                    <-- données riomètres CANOPUS (csv)
|   +-- old_canopus_riometer_format/                <-- ancien format riomètre
|
|-- carisma/                                        <-- fichiers CARISMA BRUTS (.tar)
|
|-- CANOPUS/                                        <-- fichiers CANOPUS bruts (.tar.gz)
```
Stations magnétométriques : BACK, CONT, DAWS, ESKI, FCHU, FSIM, FSMI, GILL, GULL, ISLL, MCMU, MSTK, PINA, RABB, RANK, TALO




### Explorer le serveur

In [ ]:
# %pip install pandas matplotlib

In [ ]:
import os 
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from io import StringIO
from datetime import datetime, timedelta

# Le serveur de l'ASC peut renvoyer un certificat auto-signé — désactiver la vérification SSL
requests.packages.urllib3.disable_warnings()

BASE_URL = "https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/"
MAG_BASE = BASE_URL + "CARISMA/carisma_csv/mag/daily/"

In [ ]:
def list_directory(url):
    resp = requests.get(url, verify=False)
    
    links = re.findall(r'href="([^"]+)"', resp.text)

    # Filtrer les liens vers les répertoires parents et les chaînes de requête
    names = []
    for link in links:
        name = link.strip('/').split('/')[-1]
        if name and name != ".." and "?" not in link and link != "../":
            # Ignorer les liens qui pointent vers le répertoire parent
            if not link.startswith("/"):
                names.append(name)
    return names    


# Naviguer dans l'archive des données magnétométriques
print("Années de données magnétométriques disponibles :")
years = list_directory(MAG_BASE)
print(years)

# Lister les stations pour 2008
print("\nStations disponibles en 2008 :")
stations = list_directory(MAG_BASE + "2008/")
print(stations)

# Lister les premiers fichiers pour la station GILL
print("\n5 premiers fichiers pour la station GILL en 2008 :")
files = list_directory(MAG_BASE + "2008/GILL/")
print(files[:5])


### Partie 1 : Télécharger les données magnétométriques

#### 1.1 Télécharger par programmation

In [ ]:
def download_mag_file(station, date_str, save_dir="data/mag"):
    year = date_str[:4]
    filename = f"{date_str}{station}.MAG.csv"
    url = f"{MAG_BASE}{year}/{station}/{filename}"

    os.makedirs(save_dir, exist_ok=True)
    local_path = os.path.join(save_dir, filename)

    try:
        resp = requests.get(url, verify=False)
        with open(local_path, 'wb') as f:
            f.write(resp.content)
        print(f"Téléchargé {filename} dans : {local_path}")
    except Exception as e:
        print(f"Erreur lors du téléchargement de {filename} : {e}")
        return None
    
    return local_path

# Télécharger une journée de données pour la station GILL (par ex., 20080101)
download_mag_file("GILL", "20080101")

In [ ]:
def download_mag_batch(station, start_date, end_date, save_dir="data/mag"):
    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")

    os.makedirs(save_dir, exist_ok=True)

    downloaded = []
    current = start

    while current <= end:
        date_str = current.strftime("%Y%m%d")
        year = current.strftime("%Y")
        filename = f"{date_str}{station}.MAG.csv"
        url = f"{MAG_BASE}{year}/{station}/{filename}"
        local_path = os.path.join(save_dir, filename)

        try:
            resp = requests.get(url, verify=False)
            with open(local_path, 'wb') as f:
                f.write(resp.content)
            downloaded.append(local_path)
            print(f"Téléchargé {filename} dans : {local_path}")
        except Exception as e:
            print(f"Erreur lors du téléchargement de {filename} : {e}")

        current += timedelta(days=1)

    print(f"Terminé ! {len(downloaded)} fichier(s) téléchargé(s) dans {save_dir}.")
    return downloaded

# Télécharger trois jours de données pour la station GILL (par ex., 20080101 à 20080103)
mag_files = download_mag_batch("GILL", "20080101", "20080103")

#### 1.2 Téléchargement manuel

1. Visitez d'abord le [Portail de données ouvertes de l'ASC - Jeu de données CARISMA (CSV)](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/carisma_csv/mag/daily/)
2. Parcourez ou recherchez les fichiers dont vous avez besoin
3. Téléchargez-les et enregistrez-les localement
4. Placez les fichiers dans le dossier /data/mag/ de ce tutoriel

### Partie 2 : Charger et explorer les données magnétométriques

Les fichiers CSV magnétométriques ont une structure spécifique avec des lignes de commentaire, des métadonnées de station et les données elles-mêmes.

**Aperçu du format des fichiers :**
```
#Tous les documents produits en utilisant les données CARISMA sont soumis à...
"#Les auteurs remercient I.R. Mann, D.K. Milling et le reste de l'équipe CARISMA...
#L'article de référence CARISMA suivant doit également être cité : Mann I. R. et al. (2008)...
# Site Lat Long yyyymmdd CoordSys Units no of records
GILL  56.376 265.360 20080101 GEODETIC nT  1Hz
 
# Date(dd/mm/yyyy),time(hh:mi:ss),X,Y,Z,F=. si les données sont valides, 
2008/01/01,00:00:00,10579.525,-717.959,59529.581,.
2008/01/01,00:00:01,10579.541,-717.950,59529.516,.
2008/01/01,00:00:02,10579.534,-717.947,59529.534,.
```

Les champs clés sont :
- **X** : Composante nord géographique (nT)
- **Y** : Composante est géographique (nT)
- **Z** : Composante verticale (positive vers le bas) (nT)
- **Flag** : `.` signifie données valides, tout autre valeur indique que les données sont suspectes


In [ ]:
def load_mag_data(file_path):
    data_lines = []
    metadata = {}

    with open(file_path, 'r') as f:
        for line in f:
            stripped = line.strip().strip('"')

            # Ignorer les lignes de commentaire et les lignes vides
            if stripped.startswith("#") or stripped == "":
                continue
            
            # Détecter la ligne de métadonnées de la station (pas de virgules, séparé par des espaces)
            if ',' not in stripped and len(stripped.split()) >= 6:
                parts = stripped.split()
                metadata= {
                    "station": parts[0],
                    "latitude": float(parts[1]),
                    "longitude": float(parts[2]),
                    "date": float(parts[3]),
                    "coord_system": parts[4],
                    "units": parts[5]
                }
                if len(metadata) > 6:
                    metadata['sampling'] = parts[6]
                continue

            # Sinon, c'est une ligne de données
            data_lines.append(stripped)

    # Convertir les lignes de données en DataFrame
    df = pd.read_csv(
        StringIO("\n".join(data_lines)),
        names=["date", "time", "X", "Y", "Z", "flag"],
        header=None
    )

    # Combiner la date et l'heure en une seule colonne datetime
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])

    return df, metadata

In [ ]:
# Charger une journée de données magnétométriques
mag_df, mag_meta = load_mag_data("data/mag/20080101GILL.MAG.csv")

print("Métadonnées de la station :")
for key, value in mag_meta.items():
    print(f"{key}: {value}")

print(f"\nForme du DataFrame : {mag_df.shape} (lignes, colonnes)")
print(f"Plage temporelle : {mag_df['datetime'].min()} à {mag_df['datetime'].max()}")
print(f"Intervalle d'échantillonnage : ~{mag_df['datetime'].diff().median().total_seconds():.0f} seconde(s)")

mag_df.head()

In [ ]:
# Explorer les données
print("Statistiques du champ magnétique (nT) :")
print(mag_df[["X", "Y", "Z"]].describe().round(2))

# Vérifier les indicateurs de qualité des données
valid_count = (mag_df['flag'] == '.').sum()
total_count = len(mag_df)
print(f"\nQualité des données : {valid_count}/{total_count} enregistrements valides ({valid_count/total_count*100:.2f}%)")

### Partie 3 : Visualiser les données magnétométriques

Les magnétomètres CARISMA mesurent les composantes X, Y, Z du champ magnétique terrestre dans le système de coordonnées géodésiques.

| Composante | Direction | Plage typique |
|-----------|-----------|---------------|
| **X** | Nord géographique | ~10 000 nT |
| **Y** | Est géographique | ~-1 000 nT |
| **Z** | Vertical (positif vers le bas) | ~59 000 nT |

In [ ]:
# Tracer les trois composantes du champ magnétique pour une journée
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

components = ['X', 'Y', 'Z']
colors = ['tab:blue', 'tab:orange', 'tab:green']
labels = ['X (Nord)', 'Y (Est)', 'Z (Bas)']

for ax, comp, color, label in zip(axes, components, colors, labels):
    ax.plot(mag_df['datetime'], mag_df[comp], color=color, linewidth=0.5)
    ax.set_ylabel(f"{label}\n(nT)")
    ax.grid(True, alpha=0.3)

station_name = mag_meta.get("station", "Unknown")
date_label = mag_df['datetime'].dt.date.iloc[0]
axes[0].set_title(
    f"Données magnétométriques CARISMA pour {station_name} le {date_label}",
    fontsize=14
)

axes[-1].set_xlabel("Heure (UTC)")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=3))

plt.tight_layout()
plt.savefig("magnetometer_plot.png", dpi=150, bbox_inches='tight')
plt.show()

print("Enregistré : magnetometer_plot.png")

In [ ]:
# Vue rapprochée : premières 6 heures de la composante X
cutoff = mag_df['datetime'].iloc[0] + pd.Timedelta(hours=6)
subset = mag_df[mag_df['datetime'] < cutoff]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(subset['datetime'], subset['X'], color='tab:blue', linewidth=0.7)
ax.set_label('Heure (UTC)')
ax.set_ylabel("X (Nord) [nT]")
ax.set_title(f"Zoom : composante X pour {station_name} le {date_label}", fontsize=14)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()

In [ ]:
# Tracé multi-jours : combiner tous les fichiers magnétométriques
mag_folder = "data/mag/"
all_mag_dfs = []

for f in sorted(os.listdir(mag_folder)):
    if f.endswith(".MAG.csv"):
        df, _ = load_mag_data(os.path.join(mag_folder, f))
        all_mag_dfs.append(df)

if len(all_mag_dfs) > 1:
    combined_df = pd.concat(all_mag_dfs, ignore_index=True)

    fig, ax = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

    for ax, comp, color, label in zip(ax, components, colors, labels):
        ax.plot(combined_df['datetime'], combined_df[comp], color=color, linewidth=0.3)
        ax.set_ylabel(f'{label}\n(nT)')
        ax.grid(True, alpha=0.3)

    axes[-0].set_title(
        f"Données magnétométriques CARISMA pour {station_name} (Plusieurs jours)",
        fontsize=16,
    )
    
    axes[-1].set_xlabel("Heure (UTC)")
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))

    plt.tight_layout()
    plt.show()

    plt.tight_layout()
    plt.show()